# Capítulo 22 · Laboratório 22: Market Profile e Volume Profile
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. O Yahoo não fornece dados negócio a negócio: o perfil de cada pregão é montado com barras de 60 minutos, espalhando o volume de cada barra entre a mínima e a máxima (aproximação grosseira). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap22.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O Yahoo só guarda ~730 pregões de barras de 60 minutos e ~60 pregões de barras de 5 e 30 minutos, contados de hoje. As barras do livro vão de 23/10/2024 (60 minutos) e de 17/08/2026 (5 e 30 minutos) até 30/09/2026: quando esse período sai da janela, o notebook usa o que o Yahoo ainda guarda e avisa, e as datas e os números mudam. O laboratório testa POC x VWAP, a abertura contra a área de valor de ontem, a regra dos 80%, o POC virgem e o equilíbrio inicial, e desenha um dia real em letras. A Parte 8 monta o footprint, o delta e o delta acumulado com os negócios do contrato cheio de Ibovespa futuro (arquivo CSV do hub, na pasta dados_negocios, ou a lista de negócios exportada da sua plataforma); sem o arquivo, ela é pulada.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import string
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
INI60, INI5 = "2024-10-23", "2026-08-16"  # início das barras de 60 minutos e das de 5 e 30 minutos do livro
CACHE = "dados_cap22.pkl"


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


pc = lambda x, k=0: "" if pd.isna(x) else f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=2: "" if pd.isna(x) else ("+" if x >= 0 else "") + pc(x, k)
nn = lambda x: f"{x:,}".replace(",", ".")
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ATIVOS = ["SPY", "QQQ", "IWM", "DIA", "BOVA11.SA", "PETR4.SA", "VALE3.SA", "ITUB4.SA"]
CURTOS = ["SPY", "BOVA11.SA", "PETR4.SA"]          # barras de 5 e 30 minutos para as figuras
PASSO = 0.001          # tamanho da faixa de preço do perfil: 0,1% do preço
AREA = 0.70            # área de valor: 70% do volume

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):
    H60, M5, M30, DX = pd.read_pickle(CACHE)
else:
    # O Yahoo só aceita data de início nos últimos ~730 dias (60 minutos) e ~60 dias (5 e 30 minutos). Pedindo o
    # período máximo (period), ele devolve ~730 e ~60 pregões; o corte refaz o período do livro enquanto ele couber.
    def intra(tickers, periodo, intervalo, ini):
        x = yf.download(tickers, period=periodo, interval=intervalo, auto_adjust=True, progress=False,
                        group_by="ticker")
        confere({f"as barras de {intervalo[:-1]} minutos": x})
        dia = x.index.strftime("%Y-%m-%d")
        corte = x[(dia >= ini) & (dia < FIM)]
        return corte if len(corte) else x       # o período do livro saiu da janela: fica a mais recente

    H60 = intra(ATIVOS, "730d", "60m", INI60)
    M5 = intra(CURTOS, "60d", "5m", INI5)
    M30 = intra(CURTOS, "60d", "30m", INI5)
    # diário com preços da época, para o perfil de faixa (Wyckoff) e o perfil de tendência
    DX = yf.download(["RENT3.SA", "PETR4.SA", "ITUB4.SA", "VALE3.SA"], start="2022-01-01", end=FIM, auto_adjust=False,
                     progress=False, group_by="ticker")
    confere({"o diário com preços da época": DX})
    if (f"{H60.index[0]:%Y-%m-%d}" > "2024-10-23" or f"{M5.index[0]:%Y-%m-%d}" > "2026-08-17"
            or f"{H60.index[-1]:%Y-%m-%d}" != "2026-09-30" or f"{M5.index[-1]:%Y-%m-%d}" != "2026-09-30"):
        print(f"Aviso: o Yahoo só guarda ~730 pregões de barras de 60 minutos e ~60 de barras de 5 e 30 minutos, contados"
              f" de hoje. Esta execução usa a janela que o Yahoo guarda hoje, e não a do livro (60 minutos de 23/10/2024 a"
              f" 30/09/2026; 5 e 30 minutos de 17/08/2026 a 30/09/2026): 60 minutos de {H60.index[0]:%d/%m/%Y} a"
              f" {H60.index[-1]:%d/%m/%Y} e 5 minutos de {M5.index[0]:%d/%m/%Y} a {M5.index[-1]:%d/%m/%Y}. As datas e os"
              " números diferem dos do livro.")
    pd.to_pickle((H60, M5, M30, DX), CACHE)


def barras(fonte, t):
    d = fonte[t].drop(columns=["Adj Close"], errors="ignore").dropna()
    # barras intradiárias: fica a barra com volume zero (o Yahoo informa volume zero na barra das 10h em muitos
    # pregões da B3; sem ela, o dia começaria às 11h). No diário, volume zero é dia sem negócio e sai.
    minimo = 0 if fonte is DX else -1
    d = d[(d["Volume"] > minimo) & (d["High"] >= d["Low"])]
    if d.index.tz is not None:
        d.index = d.index.tz_convert("America/Sao_Paulo" if t.endswith(".SA") else "America/New_York").tz_localize(None)
    return d


def distribui(b, passo, ref_baixo):
    """Espalha o volume de cada barra por igual entre a mínima e a máxima dela, em degraus de 'passo'."""
    lo, hi = b["Low"].min(), b["High"].max()
    niveis = np.arange(np.floor(lo / passo), np.floor(hi / passo) + 1) * passo
    vol = np.zeros(len(niveis))
    for l_, h_, v_ in b[["Low", "High", "Volume"]].to_numpy():
        dentro = (niveis >= np.floor(l_ / passo) * passo) & (niveis <= h_)
        if dentro.any():
            vol[dentro] += v_ / dentro.sum()
    return niveis, vol


def area_de_valor(vol, area=AREA):
    """Índices do POC e das pontas da área de valor: cresce a partir do POC pelo lado com mais volume."""
    poc = int(vol.argmax())
    a = z = poc
    total, acum = vol.sum(), vol[poc]
    while acum < area * total:
        cima = vol[z + 1] if z + 1 < len(vol) else -1
        baixo = vol[a - 1] if a > 0 else -1
        if cima >= baixo:
            z += 1
            acum += cima
        else:
            a -= 1
            acum += baixo
    return poc, a, z


def perfil(b, passo=None):
    """POC, área de valor (VAL, VAH) e VWAP de um dia."""
    passo = passo or b["Close"].iloc[0] * PASSO
    niveis, vol = distribui(b, passo, None)
    poc, a, z = area_de_valor(vol)
    tipico = (b["High"] + b["Low"] + b["Close"]) / 3
    vwap = (tipico * b["Volume"]).sum() / b["Volume"].sum()
    return niveis[poc] + passo / 2, niveis[a], niveis[z] + passo, vwap


dias = []
for t in ATIVOS:
    d = barras(H60, t)
    for data, x in d.groupby(d.index.date):
        if len(x) < 5:
            continue
        poc, val, vah, vwap = perfil(x)
        dias.append({"t": t, "data": pd.Timestamp(data), "abre": x["Open"].iloc[0], "fecha": x["Close"].iloc[-1],
                     "max": x["High"].max(), "min": x["Low"].min(), "poc": poc, "val": val, "vah": vah,
                     "vwap": vwap, "ib_max": x["High"].iloc[0], "ib_min": x["Low"].iloc[0],
                     "barras": x[["High", "Low", "Close"]].to_numpy()})
D = pd.DataFrame(dias).sort_values(["t", "data"]).reset_index(drop=True)
for col in ["poc", "val", "vah", "max", "min"]:
    D[f"{col}_ant"] = D.groupby("t")[col].shift(1)
D = D.dropna(subset=["poc_ant"]).reset_index(drop=True)
print(f"Parte 1. {nn(len(D))} pregões de {D['t'].nunique()} ativos em barras de 60 minutos"
      f" ({D['data'].min():%d/%m/%Y} a {D['data'].max():%d/%m/%Y}); barras de 5 e 30 minutos de"
      f" {barras(M5, 'SPY').index[0]:%d/%m/%Y} a {barras(M5, 'SPY').index[-1]:%d/%m/%Y}.")

## Parte 2: POC x VWAP

In [ ]:
dist = (D["poc"] - D["vwap"]).abs()
larg = D["vah"] - D["val"]
D["dia"] = D["fecha"] / D["abre"] - 1
D["amplitude"] = D["max"] / D["min"] - 1
print("\nParte 2. POC x VWAP no mesmo dia")
print(f"   distância mediana: {pc((dist / D['vwap']).median(), 2)} do preço | {pc((dist / larg).median())} da largura"
      f" da área de valor")
print(f"   dias em que ficaram a até 0,1% um do outro: {pc((dist <= D['vwap'] * PASSO).mean())}")
tend = D["dia"].abs() / D["amplitude"] > 0.7                    # dia de tendência: fechou perto de uma ponta
print(f"   distância mediana nos dias de tendência (abertura ao fechamento > 70% da amplitude):"
      f" {pc((dist / D['vwap'])[tend].median(), 2)} ({nn(int(tend.sum()))} dias) | nos outros dias:"
      f" {pc((dist / D['vwap'])[~tend].median(), 2)}")

## Parte 3: abertura x área de valor de ontem

In [ ]:
D["onde"] = np.select([D["abre"] > D["vah_ant"], D["abre"] < D["val_ant"]], ["acima", "abaixo"], "dentro")
print("\nParte 3. Abertura em relação à área de valor de ontem: retorno da abertura ao fechamento")
P3 = {}
for k in ["acima", "dentro", "abaixo"]:
    x = D[D["onde"] == k]
    P3[k] = (len(x), x["dia"].mean(), (x["dia"] > 0).mean())
    print(f"   abriu {k:<7} {nn(len(x)):>6} dias ({pc(len(x) / len(D))}) | dia médio {ps(x['dia'].mean())}"
          f" | fechou acima da abertura em {pc((x['dia'] > 0).mean())}")

## Parte 4: regra dos 80%

In [ ]:
casos = []
for _, r in D[D["onde"] != "dentro"].iterrows():
    b = r["barras"]                                     # colunas: máxima, mínima, fechamento
    entrou = None
    for i in range(1, len(b)):                          # precisa de uma barra fechada dentro da área de valor
        if r["val_ant"] <= b[i, 2] <= r["vah_ant"]:
            entrou = i
            break
    if entrou is None:
        continue
    resto = b[entrou:]
    cruzou = (resto[:, 1] <= r["val_ant"]).any() if r["onde"] == "acima" else (resto[:, 0] >= r["vah_ant"]).any()
    casos.append(cruzou)
REGRA80 = (len(casos), float(np.mean(casos)))
print(f"\nParte 4. Regra dos 80%: abriu fora da área de valor de ontem e fechou uma barra de 60 minutos dentro dela"
      f"\n   {nn(len(casos))} casos | atravessou até o outro lado da área de valor no mesmo dia: {pc(np.mean(casos))}")

## Parte 5: POC virgem

In [ ]:
res = []
for t, x in D.groupby("t"):
    x = x.reset_index(drop=True)
    for i in range(len(x) - 11):
        poc = x.at[i, "poc"]
        amanha = x.iloc[i + 1]
        if amanha["min"] <= poc <= amanha["max"]:
            continue                                    # revisitado no dia seguinte: não é virgem
        prox = x.iloc[i + 2:i + 12]
        volta = ((prox["min"] <= poc) & (prox["max"] >= poc)).any()
        nivel = 2 * amanha["fecha"] - poc               # controle: mesma distância do fechamento, do outro lado
        volta_ctrl = ((prox["min"] <= nivel) & (prox["max"] >= nivel)).any()
        res.append((volta, volta_ctrl, abs(poc / amanha["fecha"] - 1)))
R = np.array(res)
print(f"\nParte 5. POC virgem (não tocado no dia seguinte): {nn(len(R))} casos, a {pc(np.median(R[:, 2]), 1)} do"
      f" fechamento (mediana)")
print(f"   tocado nos 10 pregões seguintes: {pc(R[:, 0].mean())} | nível de controle à mesma distância, do outro lado:"
      f" {pc(R[:, 1].mean())}")

## Parte 6: equilíbrio inicial

In [ ]:
cima, baixo = D["max"] > D["ib_max"], D["min"] < D["ib_min"]
print("\nParte 6. Equilíbrio inicial (primeira barra de 60 minutos)")
EI = {"só a máxima": (cima & ~baixo).mean(), "só a mínima": (~cima & baixo).mean(),
      "os dois lados": (cima & baixo).mean(), "nenhum": (~cima & ~baixo).mean()}
print("   rompeu " + " | ".join(f"{k}: {pc(v)}" for k, v in EI.items()))
ganhos = []
for _, r in D[cima].iterrows():                     # compra acima da máxima do EI, stop na mínima, sai no fechamento
    b = r["barras"][1:]
    entrada, stop = r["ib_max"], r["ib_min"]
    i = int(np.argmax(b[:, 0] > entrada))
    depois = b[i:]
    if (depois[:, 1] <= stop).any():
        ganhos.append(-1.0)
    else:
        ganhos.append((r["fecha"] - entrada) / (entrada - stop))
G = np.array(ganhos)
print(f"   comprar o rompimento da máxima do EI (stop na mínima do EI, saída no fechamento): {nn(len(G))} operações,"
      f" resultado médio {nr(G.mean())} R | acerto {pc((G > 0).mean())} | stop atingido {pc((G <= -0.999).mean())}")
for m, nome in ((D["t"].str.endswith(".SA"), "B3"), (~D["t"].str.endswith(".SA"), "EUA")):
    gm = np.array([g for g, ok in zip(ganhos, m[cima].to_numpy()) if ok])
    print(f"      {nome}: {nn(len(gm))} operações, resultado médio {nr(gm.mean())} R")

## Parte 7: um dia real em letras e em volume

In [ ]:
def tpo(b30, passo):
    """Perfil em letras: cada barra de 30 minutos ganha uma letra (A, B, C...) em todos os degraus que tocou."""
    letras = string.ascii_uppercase
    lo, hi = b30["Low"].min(), b30["High"].max()
    niveis = np.arange(np.floor(lo / passo), np.floor(hi / passo) + 1) * passo
    linhas = {i: "" for i in range(len(niveis))}
    for k, (l_, h_) in enumerate(b30[["Low", "High"]].to_numpy()):
        for i, nv in enumerate(niveis):
            if nv + passo > l_ and nv <= h_:
                linhas[i] += letras[k]
    cont = np.array([len(linhas[i]) for i in range(len(niveis))], dtype=float)
    return niveis, linhas, cont


def dia_exemplo(t, data, passo, passo_tpo=0.5):
    b5 = barras(M5, t)
    b30 = barras(M30, t)
    x5, x30 = b5[b5.index.date == pd.Timestamp(data).date()], b30[b30.index.date == pd.Timestamp(data).date()]
    niveis, vol = distribui(x5, passo, None)
    p, a, z = area_de_valor(vol)
    tipico = (x5["High"] + x5["Low"] + x5["Close"]) / 3
    vwap = (tipico * x5["Volume"]).sum() / x5["Volume"].sum()
    nt, linhas, cont = tpo(x30, passo_tpo)
    pt, at, zt = area_de_valor(cont)
    return {"t": t, "data": pd.Timestamp(data), "b5": x5, "b30": x30, "niveis": niveis, "vol": vol,
            "poc": niveis[p] + passo / 2, "val": niveis[a], "vah": niveis[z] + passo, "vwap": vwap,
            "nt": nt, "linhas": linhas, "cont": cont, "poc_t": nt[pt] + passo_tpo / 2, "val_t": nt[at],
            "vah_t": nt[zt] + passo_tpo, "passo": passo, "passo_tpo": passo_tpo}


# dia com a maior distância entre o POC e a VWAP no SPY (para a figura), entre os dias completos de 5 minutos
b5s = barras(M5, "SPY")
cand = []
for data, x in b5s.groupby(b5s.index.date):
    if len(x) >= 70:
        p_, a_, z_, v_ = perfil(x, 0.25)
        cand.append((data, abs(p_ - v_) / v_, x["Close"].iloc[-1] / x["Open"].iloc[0] - 1))
C = pd.DataFrame(cand, columns=["data", "dist", "dia"])
EX_TEND = C.sort_values("dist").iloc[-1]["data"]              # dia com o POC mais longe da VWAP
EX_EQUI = C.sort_values("dist").iloc[len(C) // 2]["data"]     # dia típico
EXEMPLOS = {"de distribuição dupla": dia_exemplo("SPY", EX_TEND, 0.25), "típico": dia_exemplo("SPY", EX_EQUI, 0.25)}
print("\nParte 7. Dois dias reais do SPY (perfil de volume com barras de 5 minutos; letras com barras de 30 minutos)")
for k, e in EXEMPLOS.items():
    print(f"   dia {k} ({e['data']:%d/%m/%Y}): abertura {nr(e['b5']['Open'].iloc[0])}, fechamento"
          f" {nr(e['b5']['Close'].iloc[-1])} | POC do volume {nr(e['poc'])}, VWAP {nr(e['vwap'])}, área de valor"
          f" {nr(e['val'])} a {nr(e['vah'])} | POC das letras {nr(e['poc_t'])}, área de valor das letras"
          f" {nr(e['val_t'])} a {nr(e['vah_t'])}")

In [ ]:
import matplotlib.pyplot as plt
for k, e in EXEMPLOS.items():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4), sharey=True, gridspec_kw={"width_ratios": [3, 1]})
    e["b5"]["Close"].plot(ax=a1, color="black", title=f"SPY, dia {k} ({e['data']:%d/%m/%Y})")
    for nivel, rot, cor in ((e["poc"], "POC", "red"), (e["vwap"], "VWAP", "blue"), (e["vah"], "VAH", "gray"),
                            (e["val"], "VAL", "gray")):
        a1.axhline(nivel, color=cor, lw=0.8, label=rot)
    a1.legend()
    a2.barh(e["niveis"], e["vol"], height=e["passo"] * 0.9, color="gray")
    plt.show()
for i, nv in reversed(list(enumerate(EXEMPLOS["de distribuição dupla"]["nt"]))):
    print(f"{nv:8.2f}  {''.join(sorted(EXEMPLOS['de distribuição dupla']['linhas'][i]))}")

## Parte 8: footprint, delta e delta acumulado com os negócios do pregão

In [ ]:
# Arquivo: a lista de negócios (times & trades) exportada da sua plataforma, em CSV ou Excel, com hora, preço,
# quantidade e lado do agressor. O exemplo do livro é o contrato cheio de Ibovespa futuro (INDZ26) em 20/10/2026, das 9h00 às 15h18.
NEGOCIOS = "dados_negocios/negocios_INDZ26_20261020.csv"
URL_NEGOCIOS = ("https://raw.githubusercontent.com/traderbrasil/livro-analise-tecnica/v4.0/"
                "dados_negocios/negocios_INDZ26_20261020.csv")
if not os.path.exists(NEGOCIOS):                      # no Colab, baixa o arquivo do exemplo do repositório do livro
    try:
        import urllib.request
        os.makedirs("dados_negocios", exist_ok=True)
        urllib.request.urlretrieve(URL_NEGOCIOS, NEGOCIOS)
    except Exception as erro:
        print("Não consegui baixar o arquivo de negócios:", erro)
DIA_NEG = "2026-10-20"
COLUNAS = {"Data": "hora", "Hora": "hora", "Valor": "preco", "Preço": "preco", "Quantidade": "qtd",
           "quantidade": "qtd", "Agressor": "agressor"}


def le_negocios(arquivo, dia):
    t = pd.read_excel(arquivo) if arquivo.endswith("xlsx") else pd.read_csv(arquivo, sep=None, engine="python")
    t = t.rename(columns=COLUNAS)
    h = t["hora"].astype(str)
    t["t"] = pd.to_datetime(h, dayfirst=True) if h.str.contains("/").all() else pd.to_datetime(dia + " " + h)
    t = t.sort_values("t", kind="stable")                 # a exportação costuma vir do mais novo para o mais antigo
    t["compra"] = np.where(t["agressor"] == "Comprador", t["qtd"], 0)   # compra agredida (pagou a oferta de venda)
    t["venda"] = np.where(t["agressor"] == "Vendedor", t["qtd"], 0)     # venda agredida (aceitou a oferta de compra)
    return t        # leilão, negócio direto e RLP (sem agressor definido) contam no volume, mas não no delta


def footprint(neg, minutos=5, linha=25):
    """Barras de N minutos; em cada barra, compra e venda agredidas por faixa de preço (linha, em pontos)."""
    neg = neg.assign(barra=neg["t"].dt.floor(f"{minutos}min"), nivel=(neg["preco"] // linha) * linha)
    fp = {}
    for b, x in neg.groupby("barra"):
        tab = x.groupby("nivel")[["venda", "compra"]].sum().sort_index()
        fp[b] = {"tab": tab, "o": x["preco"].iloc[0], "h": x["preco"].max(), "l": x["preco"].min(),
                 "c": x["preco"].iloc[-1], "vol": int(x["qtd"].sum()), "delta": int((x["compra"] - x["venda"]).sum()),
                 "poc": (tab["venda"] + tab["compra"]).idxmax()}
    res = pd.DataFrame({b: {k: v[k] for k in ("o", "h", "l", "c", "vol", "delta", "poc")} for b, v in fp.items()}).T
    res["cvd"] = res["delta"].cumsum()                      # delta acumulado (CVD)
    return fp, res


def desequilibrios(tab, linha, razao=3.0, minimo=20):
    """Desequilíbrio na diagonal: compra agredida num nível contra venda agredida no nível de baixo (e o inverso)."""
    c, v = tab["compra"], tab["venda"]
    v_abaixo = v.reindex(tab.index - linha).fillna(0).to_numpy()
    c_acima = c.reindex(tab.index + linha).fillna(0).to_numpy()
    compra = (c >= razao * np.maximum(v_abaixo, 1)) & (c >= minimo)
    venda = (v >= razao * np.maximum(c_acima, 1)) & (v >= minimo)
    return compra, venda


def empilhados(marcas, n=3):
    """Níveis com n ou mais desequilíbrios seguidos do mesmo lado."""
    m, run, saida = list(marcas), 0, []
    for i, x in enumerate(m):
        run = run + 1 if x else 0
        if run >= n:
            saida.extend(range(i - run + 1, i + 1))
    return sorted(set(saida))


sn = lambda v: f"{int(v):+,}".replace(",", ".")       # inteiro com sinal e ponto de milhar


if os.path.exists(NEGOCIOS):
    NEG = le_negocios(NEGOCIOS, DIA_NEG)
    LINHA = 50
    FP, RES = footprint(NEG, 5, LINHA)
    print(f"\nParte 8. Footprint do INDZ26 em {pd.Timestamp(DIA_NEG):%d/%m/%Y} ({nr(len(NEG), 0)} negócios, das"
          f" {NEG['t'].min():%H:%M} às {NEG['t'].max():%H:%M}; barras de 5 minutos, linhas de {LINHA} pontos)")
    print(f"   compra agredida {nr(NEG['compra'].sum(), 0)} contratos, venda agredida {nr(NEG['venda'].sum(), 0)};"
          f" delta do período {sn(NEG['compra'].sum() - NEG['venda'].sum())}")
    # Os horários saem dos dados do pregão: o fundo depois do equilíbrio inicial (a primeira hora) e a máxima do dia.
    t0 = NEG["t"].min().floor("5min")
    depois_ei = RES[RES.index >= t0 + pd.Timedelta("60min")]
    b_fundo = depois_ei["l"].astype(float).idxmin()        # barra com a menor mínima depois da primeira hora
    j = RES.index.get_loc(b_fundo)
    b_max = RES["h"].astype(float).idxmax()                # barra da máxima do dia
    b_min = RES.loc[b_max:, "l"].astype(float).idxmin()    # barra da menor mínima depois da máxima
    print(f"   conferência com a plataforma (delta das 12 barras antes da máxima do dia, de"
          f" {RES.index[max(RES.index.get_loc(b_max) - 12, 0)]:%Hh%M} a {RES.index[RES.index.get_loc(b_max) - 1]:%Hh%M}):",
          list(RES["delta"].iloc[max(RES.index.get_loc(b_max) - 12, 0):RES.index.get_loc(b_max)].astype(int)))
    print(f"   fundo depois da primeira hora: barra das {b_fundo:%Hh%M}; footprint das duas barras antes dela até a"
          f" seguinte")
    for b in RES.index[max(j - 2, 0):j + 2]:
        tab = FP[b]["tab"]
        dc, dv = desequilibrios(tab, LINHA)
        print(f"   {b:%H:%M}  mín {nr(FP[b]['l'], 0)}  máx {nr(FP[b]['h'], 0)}  delta {FP[b]['delta']:+}  POC"
              f" {nr(FP[b]['poc'], 0)}  desequilíbrios de compra {list(tab.index[dc].astype(int))}"
              f"  de venda {list(tab.index[dv].astype(int))}  empilhados de compra"
              f" {[int(tab.index[i]) for i in empilhados(dc)]}")
    fundo = NEG[(NEG["t"] >= RES.index[max(j - 1, 0)]) & (NEG["t"] < b_fundo + pd.Timedelta("5min"))]
    i_min = fundo["preco"].idxmin()
    perto = fundo[fundo["preco"] <= fundo["preco"].min() + 50]
    print(f"   mínima do período: {nr(fundo.loc[i_min, 'preco'], 0)} às {fundo.loc[i_min, 't']:%H:%M:%S}; venda agredida"
          f" a até 50 pontos da mínima: {int(perto['venda'].sum())} contratos")
    a, z = RES.loc[b_max], RES.loc[b_min]
    print(f"   divergência: das {b_max:%Hh%M} às {b_min:%Hh%M} o preço foi de {nr(a['h'], 0)} (máxima) a {nr(z['l'], 0)} (mínima),"
          f" e o delta acumulado foi de {sn(a['cvd'])} para {sn(z['cvd'])}")
else:
    print("\nParte 8. Sem o arquivo de negócios (" + NEGOCIOS + "): exporte os negócios do pregão e rode de novo.")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `distribui` e `area_de_valor` da Parte 1. Mostre onde o volume de cada barra é espalhado por igual entre a mínima e a máxima, por que o POC é a moda (`vol.argmax()`) e como a área de valor cresce a partir do POC até 70% (`AREA = 0.70`)." Confira com a Parte 2, que imprime a distância mediana entre o POC e a VWAP (0,11% do preço no livro; com as barras que o Yahoo guarda hoje, as datas e os números mudam, como avisa a abertura do notebook).
2. **Adaptar:** "Na Parte 8, troque `NEGOCIOS` e `DIA_NEG` pelo arquivo de negócios de um pregão de [ATIVO], exportado da sua plataforma com as mesmas colunas do arquivo do livro (hora, preço, quantidade e agressor, com Comprador e Vendedor). Ajuste `LINHA` ao preço de [ATIVO]. Não mude `footprint` nem `desequilibrios`: os horários das linhas impressas saem dos dados do pregão (o fundo depois da primeira hora e a máxima do dia)." Antes de trocar o arquivo, rode a Parte 8 com o arquivo do livro e confira o delta de −250 na barra das 10h30, o de +595 na das 10h35 e o delta acumulado indo de +5.090 para +5.405 entre 13h40 e 15h00.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque o degrau do perfil (PASSO) de 0,1% para 0,05%. A distância entre POC e VWAP muda?
2. Na Parte 3, separe B3 e EUA. A abertura acima da área de valor ajuda em algum dos dois?
3. Na Parte 4, exija duas barras de 60 minutos fechadas dentro da área de valor. A regra dos 80% melhora?
4. Na Parte 6, troque o stop da mínima do EI pelo meio do EI. O resultado em R muda?
5. Na Parte 8, troque LINHA de 50 para 25 pontos e a razão do desequilíbrio de 3 para 4. O que some do Gráfico 22.17?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.